# Camada Silver

A camada Silver recebe a tabela da camada Bronze (o `history.csv` original, sem alterações) e entrega uma base limpa, padronizada e com os tipos corretos. É dela que as três camadas Gold partem.

O caminho seguido aqui é sempre o mesmo: ler, verificar, limpar e organizar. Cada decisão de tratamento vem acompanhada do motivo, para que outra pessoa consiga repetir o processo e chegar no mesmo resultado.

## 1. Leitura

In [1]:
import pandas as pd
import numpy as np

df_bronze = pd.read_csv('../Data/Bronze/tabela_bronze.csv')

# Trabalhamos sempre numa cópia para não alterar a base original
df = df_bronze.copy(deep=True)

print('Linhas e colunas:', df.shape)
df.head(3)

Linhas e colunas: (5634, 23)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CustomerFeedback,MonthlyIncome
0,4578-PHJYZ,Male,0,Yes,Yes,52,Yes,No,DSL,No,Yes,Yes,Yes,Yes,No,One year,Yes,Electronic check,68.75,3482.85,No,"I have been a customer with this internet provider for over four years now and I have been satisfied with the service. The DSL internet is reliable and I have had no issues with connectivity. The monthly charges of $68.75 are reasonable for the service I receive. I appreciate the convenience of paying with electronic check. Overall, I am happy with my experience and have no plans to switch providers.",6532
1,6289-CPNLD,Male,0,Yes,Yes,33,Yes,No,DSL,No,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,73.90,2405.05,Yes,"I've been a customer with this company for over 2 and a half years now, and unfortunately I have decided to churn. The DSL internet service was reliable for the most part, but the monthly charges of $73.9 were a bit steep for the quality of service I was receiving. Additionally, having to mail in a check for payment was inconvenient and outdated. I ultimately decided to switch to a different provider that offered better value and more modern payment options.",7634
2,2682-KEVRP,Female,1,No,No,22,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,20.05,417.00,No,"I have been a customer with this company for 22 months now and I have been satisfied with the service. I have not experienced any issues with billing or service interruptions. The monthly charges are affordable at $20.05 and I appreciate being able to pay by mailing a check. The lack of internet service is not a problem for me as I do not require it. Overall, I am happy with my experience as a customer and have no complaints.",3628


## 2. Verificação

Antes de mexer em qualquer coisa, olhamos a base por completo: tipos, valores ausentes, duplicados e se os valores fazem sentido entre si.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5634 entries, 0 to 5633
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        5634 non-null   str    
 1   gender            5634 non-null   str    
 2   SeniorCitizen     5634 non-null   int64  
 3   Partner           5634 non-null   str    
 4   Dependents        5634 non-null   str    
 5   tenure            5634 non-null   str    
 6   PhoneService      5634 non-null   str    
 7   MultipleLines     5634 non-null   str    
 8   InternetService   5634 non-null   str    
 9   OnlineSecurity    5634 non-null   str    
 10  OnlineBackup      5634 non-null   str    
 11  DeviceProtection  5634 non-null   str    
 12  TechSupport       5634 non-null   str    
 13  StreamingTV       5634 non-null   str    
 14  StreamingMovies   5634 non-null   str    
 15  Contract          5634 non-null   str    
 16  PaperlessBilling  5634 non-null   str    
 17  Paymen

Logo de cara aparecem dois pontos: `tenure` (tempo de assinatura em meses) foi lido como texto, o que indica algum valor não numérico na coluna, e `TotalCharges` tem 278 valores vazios.

In [3]:
resumo_ausentes = pd.DataFrame({
    'Nulos': df.isna().sum(),
    'Nulos (%)': (df.isna().mean() * 100).round(2)
})
resumo_ausentes[resumo_ausentes['Nulos'] > 0]

,Nulos,Nulos (%)
TotalCharges,278,4.93


In [4]:
print('Linhas duplicadas:', df.duplicated().sum())
print('IDs de cliente repetidos:', df['customerID'].duplicated().sum())

id_valido = df['customerID'].str.fullmatch(r'[0-9]{4}-[A-Z]{5}')
print('IDs fora do padrão 0000-AAAAA:', (~id_valido).sum())

Linhas duplicadas: 0
IDs de cliente repetidos: 0
IDs fora do padrão 0000-AAAAA: 0


Não há linhas nem clientes duplicados e todos os IDs seguem o mesmo padrão. Cada linha é um cliente diferente.

Agora olhamos os valores distintos de cada coluna de texto. É aqui que costumam aparecer erros de digitação e diferenças de maiúsculas e minúsculas.

In [5]:
colunas_texto = [c for c in df.select_dtypes(include=['object', 'string']).columns
                 if c not in ['customerID', 'CustomerFeedback', 'tenure']]

for coluna in colunas_texto:
    print(f'{coluna}: {df[coluna].value_counts(dropna=False).to_dict()}')

gender: {'Male': 2835, 'Female': 2799}
Partner: {'No': 2907, 'Yes': 2727}
Dependents: {'No': 3949, 'Yes': 1685}
PhoneService: {'Yes': 5041, 'No': 523, 'no': 40, 'yes': 30}
MultipleLines: {'No': 2704, 'Yes': 2367, 'No phone service': 563}
InternetService: {'Fiber optic': 2477, 'DSL': 1924, 'No': 1233}
OnlineSecurity: {'No': 2803, 'Yes': 1598, 'No internet service': 1233}
OnlineBackup: {'No': 2457, 'Yes': 1944, 'No internet service': 1233}
DeviceProtection: {'No': 2469, 'Yes': 1932, 'No internet service': 1233}
TechSupport: {'No': 2791, 'Yes': 1610, 'No internet service': 1233}
StreamingTV: {'No': 2219, 'Yes': 2182, 'No internet service': 1233}
StreamingMovies: {'No': 2224, 'Yes': 2177, 'No internet service': 1233}
Contract: {'Month-to-month': 3099, 'Two year': 1347, 'One year': 1188}
PaperlessBilling: {'Yes': 3327, 'No': 2307}
PaymentMethod: {'Electronic check': 1888, 'Mailed check': 1283, 'Credit card (automatic)': 1234, 'Bank transfer (automatic)': 1229}
Churn: {'No': 4130, 'Yes': 150

O único problema de escrita está em `PhoneService`, que tem "yes" e "no" em minúsculas além de "Yes" e "No". As demais colunas estão consistentes.

Em `tenure`, o valor que impede a coluna de ser numérica é a palavra "unknown":

In [6]:
tenure_numerico = pd.to_numeric(df['tenure'], errors='coerce')
df.loc[tenure_numerico.isna(), 'tenure'].value_counts()

,count
tenure,
unknown,19


### Consistência entre colunas

Algumas colunas dependem de outras. Quem não tem telefone não pode ter várias linhas, e quem não tem internet não pode ter serviços adicionais de internet. Conferimos se a base respeita essas regras.

In [7]:
sem_telefone = df['PhoneService'].str.lower() == 'no'
print('Sem telefone, mas com MultipleLines diferente de "No phone service":',
      (sem_telefone & (df['MultipleLines'] != 'No phone service')).sum())

adicionais = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
              'TechSupport', 'StreamingTV', 'StreamingMovies']
sem_internet = df['InternetService'] == 'No'
inconsistentes = (df.loc[sem_internet, adicionais] != 'No internet service').any(axis=1).sum()
print('Sem internet, mas com algum serviço adicional:', inconsistentes)

Sem telefone, mas com MultipleLines diferente de "No phone service": 0
Sem internet, mas com algum serviço adicional: 0


As regras são respeitadas em todas as linhas.

Por último, testamos uma relação que deveria existir: o valor total pago deveria ser parecido com o tempo de assinatura multiplicado pela mensalidade. Se essa relação se confirmar, ela serve para tratar os valores ausentes de forma segura.

In [8]:
validos = df['TotalCharges'].notna() & (tenure_numerico > 0)
razao = df.loc[validos, 'TotalCharges'] / (tenure_numerico[validos] * df.loc[validos, 'MonthlyCharges'])

razao.describe().round(3)

,0
count,5338.000
mean,1.000
std,0.051
min,0.689
25%,0.980
50%,1.000
75%,1.020
max,1.495


A mediana da razão é 1,00 e metade dos clientes fica entre 0,98 e 1,02. Ou seja, `TotalCharges` é praticamente `tenure × MonthlyCharges`; as pequenas diferenças vêm de reajustes de mensalidade ao longo do tempo.

Também vale olhar quem tem `TotalCharges` vazio:

In [9]:
total_vazio = df['TotalCharges'].isna()
print('Clientes com TotalCharges vazio:', total_vazio.sum())
print('Desses, com tenure = 0:', (tenure_numerico[total_vazio] == 0).sum())
print('Clientes com tenure = 0 na base inteira:', (tenure_numerico == 0).sum())
print('Clientes com tenure e TotalCharges vazios ao mesmo tempo:', (total_vazio & tenure_numerico.isna()).sum())

Clientes com TotalCharges vazio: 278
Desses, com tenure = 0: 10
Clientes com tenure = 0 na base inteira: 10
Clientes com tenure e TotalCharges vazios ao mesmo tempo: 1


Os 10 clientes com `tenure = 0` acabaram de entrar e ainda não pagaram nenhuma fatura, por isso o total é vazio. Os outros 268 vazios estão espalhados pela base, sem relação com o tempo de casa (o tempo médio desse grupo é parecido com o resto).

### Resumo da verificação

| Ponto encontrado | Quantidade | O que fazemos |
|---|---|---|
| `PhoneService` com "yes"/"no" minúsculos | 70 | Padronizar a escrita |
| `tenure` com "unknown" | 19 | Estimar pelo total pago ÷ mensalidade (1 deles também não tem total e é removido) |
| `TotalCharges` vazio com `tenure = 0` | 10 | Preencher com 0 (cliente novo) |
| `TotalCharges` vazio nos demais | 268 | Estimar por `tenure × MonthlyCharges` |
| Nomes e valores em inglês | todas | Traduzir para português |
| Tipos errados (texto em vez de número/booleano) | várias | Corrigir a tipagem |

## 3. Limpeza

In [10]:
# Remover espaços extras no começo e no fim dos textos
for coluna in df.select_dtypes(include=['object', 'string']).columns:
    df[coluna] = df[coluna].str.strip()

# Valores que representam "não informado" viram nulo de verdade
df = df.replace({'unknown': pd.NA, 'Unknown': pd.NA, 'null': pd.NA, 'NULL': pd.NA, '': pd.NA})

print('Valores ausentes por coluna após a limpeza:')
df.isna().sum()[lambda s: s > 0]

Valores ausentes por coluna após a limpeza:


,0
tenure,19
TotalCharges,278


### Tradução dos valores

Colocamos tudo em minúsculas antes de traduzir. Assim "Yes" e "yes" caem no mesmo valor e o problema de `PhoneService` é resolvido junto com a tradução.

In [11]:
sim_nao = {'yes': 'Sim', 'no': 'Não'}
servico_internet = {'yes': 'Sim', 'no': 'Não', 'no internet service': 'Sem serviço de internet'}

traducao_valores = {
    'gender': {'male': 'Masculino', 'female': 'Feminino'},
    'Partner': sim_nao,
    'Dependents': sim_nao,
    'PhoneService': sim_nao,
    'PaperlessBilling': sim_nao,
    'Churn': sim_nao,
    'MultipleLines': {'yes': 'Sim', 'no': 'Não', 'no phone service': 'Sem serviço de telefonia'},
    'InternetService': {'dsl': 'DSL', 'fiber optic': 'Fibra óptica', 'no': 'Sem serviço de internet'},
    'OnlineSecurity': servico_internet,
    'OnlineBackup': servico_internet,
    'DeviceProtection': servico_internet,
    'TechSupport': servico_internet,
    'StreamingTV': servico_internet,
    'StreamingMovies': servico_internet,
    'Contract': {'month-to-month': 'Mensal', 'one year': 'Um ano', 'two year': 'Dois anos'},
    'PaymentMethod': {
        'electronic check': 'Cheque eletrônico',
        'mailed check': 'Cheque pelo correio',
        'bank transfer (automatic)': 'Transferência bancária automática',
        'credit card (automatic)': 'Cartão de crédito automático'
    }
}

for coluna, mapa in traducao_valores.items():
    df[coluna] = df[coluna].str.lower().replace(mapa)

df['PhoneService'].value_counts()

,count
PhoneService,
Sim,5071
Não,563


### Tradução das colunas

A coluna de feedback continua com o texto em inglês. Traduzir textos livres pode mudar o sentido, e se o grupo fizer análise de sentimento mais pra frente, os modelos funcionam melhor no idioma original.

In [12]:
traducao_colunas = {
    'customerID': 'ID_Cliente',
    'gender': 'Genero',
    'SeniorCitizen': 'Idoso',
    'Partner': 'Conjuge',
    'Dependents': 'Dependentes',
    'tenure': 'Tempo_Assinatura',
    'PhoneService': 'Servico_Telefonia',
    'MultipleLines': 'Varias_Linhas',
    'InternetService': 'Servico_Internet',
    'OnlineSecurity': 'Seguranca_Online',
    'OnlineBackup': 'Backup_Online',
    'DeviceProtection': 'Protecao_Dispositivos',
    'TechSupport': 'Suporte_Tecnico',
    'StreamingTV': 'Streaming_TV',
    'StreamingMovies': 'Streaming_Filmes',
    'Contract': 'Contrato',
    'PaperlessBilling': 'Faturamento_Eletronico',
    'PaymentMethod': 'Forma_Pagamento',
    'MonthlyCharges': 'Valores_Mensais',
    'TotalCharges': 'Valores_Totais',
    'Churn': 'Status_Cancelamento',
    'CustomerFeedback': 'Feedback_Cliente',
    'MonthlyIncome': 'Renda_Mensal'
}
df = df.rename(columns=traducao_colunas)
df.columns.tolist()

['ID_Cliente', 'Genero', 'Idoso', 'Conjuge', 'Dependentes', 'Tempo_Assinatura', 'Servico_Telefonia', 'Varias_Linhas', 'Servico_Internet', 'Seguranca_Online', 'Backup_Online', 'Protecao_Dispositivos', 'Suporte_Tecnico', 'Streaming_TV', 'Streaming_Filmes', 'Contrato', 'Faturamento_Eletronico', 'Forma_Pagamento', 'Valores_Mensais', 'Valores_Totais', 'Status_Cancelamento', 'Feedback_Cliente', 'Renda_Mensal']

### Tipagem

- Números: tempo de assinatura (inteiro, em meses), valores e renda.
- Booleanos: colunas que só têm Sim/Não. `Idoso` já vinha como 0/1.
- Categorias: colunas com poucas opções fixas, como tipo de contrato e de internet.

In [13]:
df['Tempo_Assinatura'] = pd.to_numeric(df['Tempo_Assinatura']).astype('Int64')
df['Valores_Mensais'] = df['Valores_Mensais'].astype('Float64')
df['Valores_Totais'] = df['Valores_Totais'].astype('Float64')
df['Renda_Mensal'] = df['Renda_Mensal'].astype('Float64')

df['Idoso'] = df['Idoso'].astype('boolean')

colunas_booleanas = ['Conjuge', 'Dependentes', 'Servico_Telefonia',
                     'Faturamento_Eletronico', 'Status_Cancelamento']
for coluna in colunas_booleanas:
    df[coluna] = df[coluna].map({'Sim': True, 'Não': False}).astype('boolean')

colunas_categoricas = ['Genero', 'Servico_Internet', 'Varias_Linhas', 'Seguranca_Online',
                       'Backup_Online', 'Protecao_Dispositivos', 'Suporte_Tecnico',
                       'Streaming_TV', 'Streaming_Filmes', 'Contrato', 'Forma_Pagamento']
for coluna in colunas_categoricas:
    df[coluna] = df[coluna].astype('category')

df.dtypes

,0
ID_Cliente,str
Genero,category
Idoso,boolean
Conjuge,boolean
Dependentes,boolean
Tempo_Assinatura,Int64
Servico_Telefonia,boolean
Varias_Linhas,category
Servico_Internet,category
Seguranca_Online,category


## 4. Tratamento dos valores ausentes

A verificação mostrou que `Valores_Totais ≈ Tempo_Assinatura × Valores_Mensais`. Usamos essa relação para preencher os vazios, em vez de usar média ou mediana da coluna, que colocariam o mesmo valor para um cliente de 2 meses e outro de 70.

Para não perder a informação de que o valor foi estimado, criamos duas colunas de controle (`Tempo_Estimado` e `Total_Estimado`). Quem quiser fazer uma análise só com os valores originais consegue filtrar por elas.

In [14]:
# Um cliente não tem nem o tempo nem o total. Sem um dos dois não há como estimar o outro,
# então ele sai da base (é 1 cliente em 5.634, não muda nenhum resultado)
sem_base = df['Tempo_Assinatura'].isna() & df['Valores_Totais'].isna()
print('Clientes removidos por não ter tempo nem total:', sem_base.sum())
df = df[~sem_base].reset_index(drop=True)

# Tempo de assinatura desconhecido: estimado pelo total pago dividido pela mensalidade.
# O resultado é limitado a 72 meses, que é o maior tempo registrado na base.
df['Tempo_Estimado'] = df['Tempo_Assinatura'].isna()
estimativa_tempo = (df['Valores_Totais'] / df['Valores_Mensais']).round().clip(upper=72)
df.loc[df['Tempo_Estimado'], 'Tempo_Assinatura'] = estimativa_tempo[df['Tempo_Estimado']].astype('Int64')

# Total vazio: cliente novo (tempo 0) fica com 0; os demais recebem tempo x mensalidade
df['Total_Estimado'] = df['Valores_Totais'].isna()
estimativa_total = (df['Tempo_Assinatura'] * df['Valores_Mensais']).round(2)
df.loc[df['Total_Estimado'], 'Valores_Totais'] = estimativa_total[df['Total_Estimado']]

print('Tempos estimados:', df['Tempo_Estimado'].sum())
print('Totais estimados:', df['Total_Estimado'].sum())
print('Ausentes restantes (sem contar o feedback):', df.drop(columns='Feedback_Cliente').isna().sum().sum())

Clientes removidos por não ter tempo nem total: 1
Tempos estimados: 18
Totais estimados: 277
Ausentes restantes (sem contar o feedback): 0


## 5. Validação final

Conferimos o número de clientes, se ainda sobrou algum ausente e se os valores estão dentro de faixas possíveis.

In [15]:
print('Linhas e colunas:', df.shape)
print('Tempo de assinatura entre', df['Tempo_Assinatura'].min(), 'e', df['Tempo_Assinatura'].max(), 'meses')
print('Mensalidade entre', df['Valores_Mensais'].min(), 'e', df['Valores_Mensais'].max())
print('Taxa geral de cancelamento:', f"{df['Status_Cancelamento'].mean():.1%}")

df.describe().round(2)

Linhas e colunas: (5633, 25)
Tempo de assinatura entre 0 e 72 meses
Mensalidade entre 18.4 e 118.65
Taxa geral de cancelamento: 26.7%


,Tempo_Assinatura,Valores_Mensais,Valores_Totais,Renda_Mensal
count,5633.0,5633.0,5633.0,5633.0
mean,32.33,64.6,2272.09,5104.95
std,24.5,30.15,2258.36,2839.48
min,0.0,18.4,0.0,100.0
25%,9.0,35.15,402.6,2693.0
50%,29.0,70.3,1390.6,5084.0
75%,55.0,89.8,3756.4,7575.0
max,72.0,118.65,8684.8,9993.0


A base terminou com 5.633 clientes (só 1 removido), sem valores ausentes, com nomes em português e tipos corretos. A taxa geral de cancelamento é de 26,7%, ou seja, pouco mais de 1 em cada 4 clientes saiu. Esse número é a referência usada nas camadas Gold para dizer se um grupo cancela mais ou menos que a média.

In [16]:
df.to_csv('../Data/Silver/tabela_silver.csv', index=False, encoding='utf-8-sig')
print('Camada Silver salva em ../Data/Silver/tabela_silver.csv')

Camada Silver salva em ../Data/Silver/tabela_silver.csv
